# 06 · Correlated noise

Chapter 05 treated every InSAR pixel as carrying its own independent error.
Real radar noise is not like that: neighboring pixels see almost the same
atmosphere and share almost the same error. This chapter shows why that
matters, how to describe the correlation, and what it costs to ignore it. The
headline result is a warning about over-confidence: pretend correlated pixels
are independent and your error bars come out far too small.

**Learning objectives**

By the end of the chapter, you will be able to:

- explain why InSAR noise is correlated in space
- write a covariance function and build a covariance matrix with GeoDef
- estimate how many *independent* measurements a correlated dataset really
  contains
- compare inversions and uncertainties under diagonal and full covariance

**Prerequisites:** Chapter 05; the idea of a covariance matrix
**Estimated time:** 45–75 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Why nearby pixels are not independent

The dominant noise in an interferogram comes from turbulent water vapor in the
atmosphere and small errors in the satellite's orbit. Both vary *smoothly*
across a scene. Two pixels a kilometer apart look through nearly the same
column of atmosphere, so their noise is nearly the same. Two pixels a hundred
kilometers apart see different weather, so their noise is nearly independent.

The diagonal covariance of Chapter 05 assumes every pixel is independent. When
the noise is actually smooth, that assumption badly **overcounts** the
information. A patch of four hundred correlated pixels does not provide four
hundred independent constraints; it might provide the information of only a
handful. Treating it as four hundred makes the data look far more informative
than they are.

## 2. A covariance function

To do better we need to say, quantitatively, how the correlation between two
pixels falls off with the distance $r$ between them. A simple and widely used
choice is the **exponential** model,

$$ C(r) = \sigma^2 \, e^{-r/L}. $$

Two numbers control it. The **sill** $\sigma^2$ is the variance, the
covariance of a pixel with itself at $r = 0$. The **correlation length** $L$
is the distance over which correlation falls to a fraction $1/e$ (about 37%) of
its peak: small $L$ means the noise decorrelates quickly, large $L$ means broad,
smooth noise.

To this we usually add a **nugget**, a small amount of genuinely
pixel-by-pixel white noise placed only on the diagonal, representing effects
like thermal noise that do not correlate between neighbors.
`geodef.data.spatial_covariance` builds the full matrix from pixel coordinates,
the sill, the correlation length, and an optional nugget.

## 3. Rebuild the fault and add a correlated InSAR track

We keep the shared megathrust scenario for the fault and true slip, then build
a dense InSAR track over it. The setup cell is the familiar one from
Chapter 05.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The shared megathrust and its true smooth thrust asperity.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

Now an 18 by 18 grid of InSAR pixels with a single ascending-track look vector,
just as in Chapter 05.

In [ ]:
pix_lon, pix_lat = np.meshgrid(np.linspace(99.6, 100.4, 18),
                               np.linspace(-0.35, 0.35, 18))
pix_lon, pix_lat = pix_lon.ravel(), pix_lat.ravel()
n_pixels = pix_lon.size
look_e = np.full(n_pixels, -0.38)
look_n = np.full(n_pixels, 0.09)
look_u = np.full(n_pixels, 0.92)

ue, un, uu = fault.displacement(pix_lat, pix_lon,
                                slip_strike=slip_true[:N], slip_dip=slip_true[N:])
los_clean = look_e * ue + look_n * un + look_u * uu

## 4. Build the covariance and draw correlated noise

Here is the new step. Instead of drawing independent noise per pixel, we build
a full covariance matrix and draw one **correlated** noise field from it. We
use 5 mm of correlated noise, a 30 km correlation length, and a 2 mm nugget.

In [ ]:
C_full = geodef.data.spatial_covariance(
    pix_lon, pix_lat,
    sill=0.005**2,               # 5 mm correlated noise
    correlation_length=30_000.0, # 30 km
    model="exponential",
    nugget=0.002**2,             # 2 mm white noise on the diagonal
)

`rng.multivariate_normal` draws a noise vector whose pixels are correlated
exactly as the covariance prescribes. Adding it to the clean LOS signal gives
our noisy observations. The per-pixel standard deviation is the square root of
the covariance diagonal.

In [ ]:
noise = rng.multivariate_normal(np.zeros(n_pixels), C_full)
los = los_clean + noise
sigma_pixel = np.sqrt(np.diag(C_full))
print(f"{n_pixels} pixels; per-pixel noise std {sigma_pixel.mean() * 1000:.1f} mm")

## 5. What the correlation looks like

Dividing the covariance by the outer product of the standard deviations gives
the **correlation matrix**, whose diagonal is one. Bright off-diagonal bands
mean nearby pixels move together. A diagonal covariance would keep only the
bright diagonal line and discard everything else.

In [ ]:
correlation = C_full / np.outer(sigma_pixel, sigma_pixel)
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(correlation, cmap="magma", vmin=0, vmax=1)
ax.set(title="InSAR noise correlation matrix",
       xlabel="pixel index", ylabel="pixel index")
fig.colorbar(image, ax=ax, label="correlation")
plt.show()

## 6. How many independent measurements is that, really?

We can put a number on the overcounting. If the pixels were independent, the
dataset would carry $n$ independent measurements. Correlation reduces that to
an **effective number of independent observations**, estimated by

$$ N_{\text{eff}} \approx
   \frac{(\operatorname{tr} C)^2}{\operatorname{tr}(C^2)}, $$

where $\operatorname{tr}$ (the trace) sums the diagonal. When every pixel is
independent this formula returns $n$ exactly; as correlation spreads the
information across fewer independent modes, it drops. The calculation is two
short lines.

In [ ]:
trace_C = np.trace(C_full)
trace_C2 = np.trace(C_full @ C_full)
n_eff = trace_C**2 / trace_C2
print(f"actual pixels:                 {n_pixels}")
print(f"effective independent pixels:  {n_eff:.0f}")

The 324 pixels carry the information of far fewer truly independent ones. Any
method that assumes all 324 are independent will believe the data are much more
informative than they are, and that belief shows up directly in the error bars.

## 7. Diagonal versus full covariance

Now invert the *same* noisy data two ways. The first dataset declares only the
per-pixel standard deviations, so GeoDef assumes independent noise (a diagonal
covariance). The second passes the full covariance matrix we built. Both use
the same smoothing.

In [ ]:
insar_diag = geodef.data.insar(
    lon=pix_lon, lat=pix_lat, los=los, sigma=sigma_pixel,
    look_e=look_e, look_n=look_n, look_u=look_u,
)
insar_full = geodef.data.insar(
    lon=pix_lon, lat=pix_lat, los=los, sigma=sigma_pixel,
    look_e=look_e, look_n=look_n, look_u=look_u,
    covariance=C_full,
)

In [ ]:
settings = dict(components="dip", regularization="laplacian",
                regularization_strength=1.0)
result_diag = geodef.solve(fault, insar_diag, **settings)
result_full = geodef.solve(fault, insar_full, **settings)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
geodef.plot.slip(fault, result_diag.dip_slip, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="Diagonal C_d")
geodef.plot.slip(fault, result_full.dip_slip, ax=axes[2], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="Full C_d")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8, label="Slip (m)")
plt.show()

The two slip estimates look similar. That is expected: correlated noise mostly
shifts the estimate around rather than transforming it. The dramatic difference
is not in the slip; it is in how confident we are allowed to be about it.

## 8. The real cost: over-confident uncertainty

`geodef.invert.model_uncertainty` returns the 1-sigma slip error per patch.
Compare the two covariance assumptions, and focus on the *ratio* between them
rather than the raw values.

In [ ]:
unc_diag = geodef.invert.model_uncertainty(result_diag, fault, insar_diag)
unc_full = geodef.invert.model_uncertainty(result_full, fault, insar_full)
print(f"mean 1-sigma slip uncertainty, diagonal C_d: {unc_diag.mean() * 100:.0f} cm")
print(f"mean 1-sigma slip uncertainty, full     C_d: {unc_full.mean() * 100:.0f} cm")
print(f"the diagonal assumption looks "
      f"{unc_full.mean() / unc_diag.mean():.1f}x more certain than it should")

Two things to read here. First, both uncertainties are large in absolute terms,
several tens of centimeters or more. That is not about the covariance; it is
because a single InSAR look direction over a narrow swath constrains slip on a
deep megathrust only weakly, exactly the under-determination Chapter 05
demonstrated. Second, and this is the point of the chapter, the diagonal
assumption reports uncertainties several times *smaller* than the full
covariance does. That is the over-confidence we predicted: by treating every
correlated pixel as an independent vote, the diagonal covariance manufactures
certainty the data do not support, matching the small effective number of
independent pixels from Section 6.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
umax = max(unc_diag.max(), unc_full.max())
geodef.plot.uncertainty(fault, unc_diag, ax=axes[0], vmax=umax,
                        title="Uncertainty (diagonal C_d, over-confident)")
geodef.plot.uncertainty(fault, unc_full, ax=axes[1], vmax=umax,
                        title="Uncertainty (full C_d, honest)")
plt.tight_layout()
plt.show()

On the shared color scale, the full-covariance panel is uniformly brighter:
larger, more honest error bars everywhere. In a real study you would combine
this InSAR track with GNSS and a second look direction to shrink the absolute
uncertainty; the correlated-noise correction shown here would still be needed
to keep whatever uncertainty remains honest.

**Checkpoint.** The diagonal and full inversions used identical data and
identical smoothing. Why did only the error bars change so much, and not the
slip map?

<details><summary>Answer</summary>

The slip estimate depends mainly on the *pattern* of the data, which both
covariances see the same way. The uncertainty depends on how much *independent
information* the data carry, and that is exactly what the covariance encodes.
The diagonal version overcounts independent information, so it under-reports
uncertainty without much changing the best-fit slip.

</details>

## Checkpoint questions

**What does the correlation length $L$ mean physically?**

<details><summary>Answer</summary>

The distance over which noise stays substantially correlated. At a separation
of $L$, the exponential correlation has dropped to about 37% of its peak.
Longer $L$ means broader, smoother noise structures and fewer effectively
independent measurements.

</details>

**Why add a nugget instead of only using the smooth correlated term?**

<details><summary>Answer</summary>

Some noise really is independent from pixel to pixel (instrument and
processing effects). The nugget places that white-noise variance on the
diagonal only, keeping it separate from the spatially correlated sill so the
model does not misattribute small-scale scatter to broad atmospheric
structure.

</details>

**Does using the full covariance guarantee correct uncertainties?**

<details><summary>Answer</summary>

No. It gives correct uncertainties only to the extent the covariance *model* is
correct. Choosing the wrong correlation length or model shape gives wrong error
bars, just of a different kind than the diagonal assumption.

</details>

## Common mistakes

- **Inventing off-diagonal terms carelessly.** A covariance matrix must be
  symmetric and positive definite, or the whitening step is impossible. Build
  it from a valid model like `spatial_covariance` rather than hand-editing
  entries.
- **Confusing the nugget with the sill.** The nugget is small-scale white
  noise; the sill is the correlated variance. Merging them misstates how much
  of the noise is spatially structured.
- **Estimating covariance from post-fit residuals.** Residuals contain both
  noise and any model error, so fitting a covariance to them can absorb real
  signal. Estimate the noise covariance from far-field or pre-event data where
  possible.

## Recap

- InSAR noise is smooth in space, so nearby pixels are correlated rather than
  independent.
- A covariance function (sill, correlation length, nugget) turns that idea into
  a matrix, built by `spatial_covariance`.
- Correlation lowers the effective number of independent observations, which
  $N_{\text{eff}}$ quantifies.
- Ignoring correlation barely changes the slip but makes the reported
  uncertainties far too small.

Chapter 07 turns from the noise model to the model itself, encoding hard
physical limits such as "slip cannot reverse" that regularization cannot
express.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/06_correlated_noise_solutions.ipynb`.

1. **Correlation length sweep.** Vary the correlation length and plot the
   effective number of independent pixels against it. Which direction makes the
   data least informative?
2. **The wrong length.** Invert with half and twice the true correlation
   length and describe how the uncertainties respond.
3. **Exponential versus Gaussian.** Compare the two covariance shapes at a
   correlation length chosen so they decay over a similar range, and note where
   they differ.
4. **Challenge: two scales.** Sum two exponential covariances with different
   correlation lengths (a short atmospheric scale and a long orbital ramp) and
   predict which features of the slip each one downweights.

## Further reading

- Hanssen, R. F. (2001), *Radar Interferometry: Data Interpretation and Error
  Analysis*, on the spatial structure of interferometric noise.
- Cressie, N. (1993), *Statistics for Spatial Data*, on covariance functions
  and variograms.
- Tarantola, A. (2005), *Inverse Problem Theory*, on data covariance in
  inversion.
- [GeoDef data documentation](../docs/data.md) for `spatial_covariance` and
  the dataset covariance interface.
- The next course chapter is `07_bounds_and_constraints.ipynb`.